# Preparación del dataset a nivel de cliente

Este notebook transforma los registros diarios, organizados por cliente y tipo de uso, en un dataset analítico que contiene una única fila por cliente.

Durante la preparación se conservan las particiones originales de entrenamiento y prueba. Las variables predictoras se construyen a partir de los registros de actividad, sin incorporar directamente la variable objetivo. Esta separación reduce el riesgo de fugas de información, aunque no permite descartarlas por completo, especialmente en su dimensión temporal, debido a las limitaciones de la documentación original sobre la construcción de la etiqueta churn.

In [ ]:
from pathlib import Path

RUTA_CUADERNO = Path.cwd()
RUTA_PROYECTO = RUTA_CUADERNO.parent
RUTA_DATOS = RUTA_PROYECTO / "datos" / "Dataset"
RUTA_RESULTADOS = RUTA_PROYECTO / "resultados"
RUTA_MODELOS = RUTA_PROYECTO / "modelos"
RUTA_EVIDENCIAS = RUTA_PROYECTO / "evidencias"

rutas_requeridas = {
    "dataset": RUTA_DATOS,
    "resultados": RUTA_RESULTADOS,
    "modelos": RUTA_MODELOS,
    "evidencias": RUTA_EVIDENCIAS
}

for nombre, ruta in rutas_requeridas.items():
    print(f"{nombre:<12} | {ruta} | existe: {ruta.exists()}")

## Carga de las particiones de entrenamiento y prueba

Se cargan los archivos correspondientes a las particiones de entrenamiento y prueba, que contienen los identificadores de los clientes y la variable objetivo `churn`.

Ambas particiones se conservarán sin modificaciones durante todo el proceso. El conjunto de entrenamiento se utilizará para construir y validar los modelos, mientras que el conjunto de prueba permanecerá separado hasta la evaluación final.

In [ ]:
import pandas as pd

RUTA_TRAIN = RUTA_DATOS / "train" / "train_cxid.csv"
RUTA_TEST = RUTA_DATOS / "test" / "test_cxid.csv"

train_clientes = pd.read_csv(RUTA_TRAIN)
test_clientes = pd.read_csv(RUTA_TEST)

print("Conjunto de entrenamiento")
print(f"Filas: {len(train_clientes):,}")
print(f"Columnas: {train_clientes.columns.tolist()}")

print("\nConjunto de prueba")
print(f"Filas: {len(test_clientes):,}")
print(f"Columnas: {test_clientes.columns.tolist()}")

## Validación de las particiones de clientes

Se comprueba que los conjuntos de entrenamiento y prueba contienen identificadores de cliente únicos, no presentan valores nulos y no comparten clientes entre sí.

Además, se verifica que la variable objetivo `churn` tiene carácter binario y únicamente contiene los valores `0` y `1`.

In [ ]:
resumen_particiones = pd.DataFrame({
    "conjunto": ["train", "test"],
    "filas": [
        len(train_clientes),
        len(test_clientes)
    ],
    "clientes_unicos": [
        train_clientes["customer_id"].nunique(),
        test_clientes["customer_id"].nunique()
    ],
    "duplicados_customer_id": [
        train_clientes["customer_id"].duplicated().sum(),
        test_clientes["customer_id"].duplicated().sum()
    ],
    "nulos_customer_id": [
        train_clientes["customer_id"].isna().sum(),
        test_clientes["customer_id"].isna().sum()
    ],
    "nulos_churn": [
        train_clientes["churn"].isna().sum(),
        test_clientes["churn"].isna().sum()
    ]
})

display(resumen_particiones)

clientes_train = set(train_clientes["customer_id"])
clientes_test = set(test_clientes["customer_id"])

clientes_compartidos = clientes_train.intersection(clientes_test)

valores_churn_train = sorted(
    train_clientes["churn"].dropna().unique().tolist()
)

valores_churn_test = sorted(
    test_clientes["churn"].dropna().unique().tolist()
)

print(
    "Clientes presentes simultáneamente en train y test:",
    len(clientes_compartidos)
)

print(
    "Valores únicos de churn en train:",
    valores_churn_train
)

print(
    "Valores únicos de churn en test:",
    valores_churn_test
)

print(
    "¿Las particiones son válidas?",
    (
        resumen_particiones["duplicados_customer_id"].eq(0).all()
        and resumen_particiones["nulos_customer_id"].eq(0).all()
        and resumen_particiones["nulos_churn"].eq(0).all()
        and len(clientes_compartidos) == 0
        and valores_churn_train == [0, 1]
        and valores_churn_test == [0, 1]
    )
)

## Construcción del universo de clientes

Se integran los identificadores de las particiones de entrenamiento y prueba en una única tabla maestra, que define el universo de clientes del proyecto.

Para cada cliente, esta tabla conserva la variable objetivo `churn` y la partición original a la que pertenece. Esta información permite mantener la separación entre los conjuntos de entrenamiento y prueba durante todo el proceso y evitar fugas de información que puedan sesgar la evaluación final.

In [ ]:
train_base = train_clientes.copy()
train_base["particion"] = "train"

test_base = test_clientes.copy()
test_base["particion"] = "test"

clientes_base = pd.concat(
    [train_base, test_base],
    ignore_index=True
)

clientes_base = clientes_base[
    ["customer_id", "churn", "particion"]
].sort_values("customer_id").reset_index(drop=True)

display(clientes_base.head())

print(f"Filas totales: {len(clientes_base):,}")
print(
    "Clientes únicos:",
    f"{clientes_base['customer_id'].nunique():,}"
)
print(
    "Duplicados de customer_id:",
    clientes_base["customer_id"].duplicated().sum()
)
print("\nDistribución por partición:")
print(clientes_base["particion"].value_counts())

## Almacenamiento de la tabla maestra de clientes

La tabla maestra de clientes se guarda en formato CSV dentro de la carpeta `resultados`.

Este archivo se utilizará como referencia común en las etapas posteriores de ingeniería de variables, entrenamiento de modelos y evaluación final.

In [ ]:
RUTA_CLIENTES_BASE = (
    RUTA_RESULTADOS / "clientes_base.csv"
)

clientes_base.to_csv(
    RUTA_CLIENTES_BASE,
    index=False
)

print("Archivo guardado en:")
print(RUTA_CLIENTES_BASE)
print(
    "¿Existe el archivo?",
    RUTA_CLIENTES_BASE.exists()
)
print(
    "Tamaño del archivo:",
    f"{RUTA_CLIENTES_BASE.stat().st_size / (1024 ** 2):,.2f} MB"
)

## Preparación de las fuentes de actividad

Se identifican los doce archivos de actividad a partir de los cuales se construirán las variables predictoras.

Estos archivos se procesarán de forma secuencial para evitar cargar simultáneamente en memoria el conjunto completo de datos de actividad.

In [ ]:
RUTA_USAGE = RUTA_DATOS / "usage_data"

archivos_actividad = sorted(
    RUTA_USAGE.glob("*.csv")
)

inventario_actividad = pd.DataFrame({
    "archivo": [
        ruta.name
        for ruta in archivos_actividad
    ],
    "usage_type": [
        ruta.stem
        for ruta in archivos_actividad
    ],
    "tamaño_mb": [
        ruta.stat().st_size / (1024 ** 2)
        for ruta in archivos_actividad
    ]
})

display(inventario_actividad)

print(
    "Número de archivos de actividad:",
    len(archivos_actividad)
)

## Verificación del orden temporal de las columnas diarias

Antes de construir las variables temporales, se examina el archivo original `full-anonymized-data.txt` para comprobar si proporciona información que permita establecer el orden cronológico de las columnas comprendidas entre `Day_1` y `Day_90`.

Dado que el archivo contiene registros anonimizados, pero no una descripción metodológica de la secuencia temporal, esta revisión no permite confirmar documentalmente el significado cronológico de los días. Por ello, el estudio adopta explícitamente un criterio de ordenación temporal, cuya interpretación y limitaciones se detallan a continuación.

In [ ]:
RUTA_DOCUMENTACION = (
    RUTA_DATOS / "raw_dump" / "full-anonymized-data.txt"
)

print("Ruta de la documentación:")
print(RUTA_DOCUMENTACION)
print("¿Existe el archivo?", RUTA_DOCUMENTACION.exists())

with open(
    RUTA_DOCUMENTACION,
    mode="r",
    encoding="utf-8",
    errors="replace"
) as archivo:
    lineas_documentacion = archivo.readlines()

print(f"Número total de líneas: {len(lineas_documentacion):,}")

### Búsqueda de referencias temporales

Se examina el archivo `full-anonymized-data.txt` para localizar posibles referencias a las columnas diarias, al periodo de observación y a su significado temporal.

Sin embargo, la presencia de estos términos en los registros no permite confirmar el orden cronológico de las observaciones ni establecer cuándo se definió la variable `churn`. En consecuencia, la interpretación temporal adoptada en el estudio debe considerarse un supuesto metodológico, tal como se explica en el apartado siguiente.

In [ ]:

terminos_busqueda = [
    "Day_1",
    "Day_90",
    "day 1",
    "day 90",
    "90 days",
    "daily",
    "observation",
    "period",
    "window"
]

# Normalizar los términos de búsqueda
terminos_minusculas = [
    termino.lower()
    for termino in terminos_busqueda
]

# Contar coincidencias sin almacenar registros individuales
coincidencias_total = 0

for linea in lineas_documentacion:
    linea_minusculas = linea.lower()

    if any(
        termino in linea_minusculas
        for termino in terminos_minusculas
    ):
        coincidencias_total += 1

print(
    f"Coincidencias encontradas: {coincidencias_total}"
)


### Conclusión sobre el orden temporal

El archivo `full-anonymized-data.txt` contiene registros anonimizados del dataset, pero no proporciona documentación descriptiva. Por tanto, no permite identificar fechas concretas ni determinar con exactitud el momento administrativo en el que se registra el `churn`.

Ante esta limitación, se adopta el orden secuencial de las columnas: `Day_1` representa el inicio del periodo de observación y `Day_90`, su final. De acuerdo con este criterio, las ventanas iniciales de 30 y 60 días comprenden las columnas de `Day_1` a `Day_30` y de `Day_1` a `Day_60`, respectivamente.

Esta interpretación permite calcular tendencias y medidas de recencia dentro de cada ventana. En este contexto, la recencia se mide respecto al último día de la ventana analizada, no respecto a una fecha de calendario conocida.

In [ ]:
COLUMNAS_DIAS = [
    f"Day_{dia}"
    for dia in range(1, 91)
]

VENTANA_INICIAL = 90

columnas_ventana = COLUMNAS_DIAS[:VENTANA_INICIAL]

print("Número de columnas diarias:", len(COLUMNAS_DIAS))
print("Ventana inicial:", VENTANA_INICIAL)
print("Primera columna:", columnas_ventana[0])
print("Última columna:", columnas_ventana[-1])

## Definición del esquema de variables conductuales

Los registros originales contienen una secuencia diaria de actividad para cada combinación de cliente y tipo de uso. Sin embargo, los modelos tabulares necesitan una única fila por cliente y un número fijo de columnas predictoras. Por este motivo, cada secuencia temporal se transforma en un conjunto reducido de indicadores conductuales.

El objetivo de esta fase es conservar información sobre cinco dimensiones principales del comportamiento:

1. **Intensidad:** cantidad total de actividad y máximo diario observado.
2. **Frecuencia:** número de días en los que existió actividad positiva.
3. **Variabilidad:** grado de fluctuación de la actividad diaria.
4. **Evolución temporal:** diferencia entre el inicio y el final de la ventana, cambio proporcional y pendiente lineal.
5. **Recencia:** actividad registrada en los últimos días y tiempo transcurrido desde la última actividad.

Para cada perfil, categoría o agregado global se calcularán exactamente once variables:

- `total`: suma de actividad durante toda la ventana;
- `std`: desviación estándar de la actividad diaria;
- `max`: valor máximo registrado en un día;
- `active_days`: número de días con actividad positiva;
- `first_third`: actividad acumulada durante el primer tercio;
- `delta_last_first`: diferencia entre el último y el primer tercio;
- `log_ratio_last_first`: diferencia logarítmica entre el último y el primer tercio;
- `slope`: pendiente lineal de la actividad a lo largo del tiempo;
- `recency`: días transcurridos desde la última actividad positiva;
- `last7`: actividad acumulada durante los últimos siete días;
- `last14`: actividad acumulada durante los últimos catorce días.

El último tercio se calcula internamente porque resulta necesario para obtener `delta_last_first` y `log_ratio_last_first`, pero no se conserva como una variable independiente. Esta decisión permite reproducir el esquema técnico utilizado en el proyecto.

Las funciones no utilizan la variable `churn`. Por tanto, la construcción de características se realiza exclusivamente a partir del comportamiento observado y no incorpora directamente información de la etiqueta.

In [ ]:
import numpy as np


def calcular_pendiente(valores):
    """
    Calcula la pendiente lineal de cada fila.

    Parámetros
    ----------
    valores : numpy.ndarray
        Matriz con clientes en filas y días en columnas.

    Devuelve
    --------
    numpy.ndarray
        Una pendiente por cliente.
    """
    numero_dias = valores.shape[1]

    eje_temporal = np.arange(
        1,
        numero_dias + 1,
        dtype=np.float64
    )

    eje_centrado = (
        eje_temporal - eje_temporal.mean()
    )

    denominador = np.sum(
        eje_centrado ** 2
    )

    valores_float = valores.astype(
        np.float64,
        copy=False
    )

    valores_centrados = (
        valores_float
        - valores_float.mean(
            axis=1,
            keepdims=True
        )
    )

    return (
        valores_centrados @ eje_centrado
    ) / denominador


def calcular_recencia(valores):
    """
    Calcula los días transcurridos desde la última
    actividad positiva hasta el final de la ventana.

    Si un cliente no presenta ninguna actividad,
    la recencia será igual al número de días observados.
    """
    actividad = valores > 0
    numero_dias = valores.shape[1]

    posiciones = np.where(
        actividad,
        np.arange(
            1,
            numero_dias + 1
        ),
        0
    )

    ultima_posicion = posiciones.max(axis=1)

    return np.where(
        ultima_posicion == 0,
        numero_dias,
        numero_dias - ultima_posicion
    )


def resumir_actividad(valores):
    """
    Transforma una matriz diaria en once variables
    conductuales por cliente.
    """
    valores = valores.astype(
        np.float64,
        copy=False
    )

    numero_dias = valores.shape[1]

    fin_primer_tercio = max(
        1,
        numero_dias // 3
    )

    inicio_ultimo_tercio = max(
        2,
        2 * numero_dias // 3
    )

    primer_tercio = valores[
        :,
        :fin_primer_tercio
    ]

    ultimo_tercio = valores[
        :,
        inicio_ultimo_tercio:
    ]

    total = valores.sum(axis=1)

    desviacion = valores.std(axis=1)

    maximo = valores.max(axis=1)

    dias_activos = (
        valores > 0
    ).sum(axis=1)

    total_primer_tercio = (
        primer_tercio.sum(axis=1)
    )

    total_ultimo_tercio = (
        ultimo_tercio.sum(axis=1)
    )

    diferencia_tercios = (
        total_ultimo_tercio
        - total_primer_tercio
    )

    ratio_logaritmico = (
        np.log1p(total_ultimo_tercio)
        - np.log1p(total_primer_tercio)
    )

    pendiente = calcular_pendiente(
        valores
    )

    recencia = calcular_recencia(
        valores
    )

    ultimos_7 = valores[
        :,
        -min(7, numero_dias):
    ].sum(axis=1)

    ultimos_14 = valores[
        :,
        -min(14, numero_dias):
    ].sum(axis=1)

    return np.column_stack([
        total,
        desviacion,
        maximo,
        dias_activos,
        total_primer_tercio,
        diferencia_tercios,
        ratio_logaritmico,
        pendiente,
        recencia,
        ultimos_7,
        ultimos_14
    ])


NOMBRES_RESUMEN = [
    "total",
    "std",
    "max",
    "active_days",
    "first_third",
    "delta_last_first",
    "log_ratio_last_first",
    "slope",
    "recency",
    "last7",
    "last14"
]

print(
    "Variables calculadas por ámbito:",
    len(NOMBRES_RESUMEN)
)

print(NOMBRES_RESUMEN)

### Validación del esquema corregido mediante una muestra

Antes de procesar los doce archivos de actividad completos, la función se aplica a una muestra de cinco clientes procedentes de uno de ellos. Esta prueba permite comprobar que la matriz diaria tiene las dimensiones esperadas y que la transformación genera exactamente once variables por cliente.

Además, se valida de manera independiente el cálculo de `log_ratio_last_first`. Para ello, se obtiene la actividad correspondiente al primer y al último tercio del periodo y se comprueba que el cálculo directo coincide con el resultado generado por la función.

El uso de una muestra pequeña permite detectar posibles errores de implementación sin incurrir en un consumo innecesario de memoria o tiempo de procesamiento. Se espera obtener una matriz de entrada de dimensiones 5 × 90, una matriz transformada de 5 × 11 y una coincidencia numérica completa en la validación de `log_ratio_last_first`.

In [ ]:
RUTA_EJEMPLO = archivos_actividad[0]

muestra_actividad = pd.read_csv(
    RUTA_EJEMPLO,
    nrows=5
)

valores_muestra = (
    muestra_actividad[columnas_ventana]
    .to_numpy(dtype=np.float32)
)

resumen_muestra = resumir_actividad(
    valores_muestra
)

columnas_resumen_muestra = [
    f"{RUTA_EJEMPLO.stem}__{nombre}"
    for nombre in NOMBRES_RESUMEN
]

resumen_muestra_df = pd.DataFrame(
    resumen_muestra,
    columns=columnas_resumen_muestra
)

resumen_muestra_df.insert(
    0,
    "customer_id",
    muestra_actividad[
        "customer_id"
    ].to_numpy()
)

numero_dias = valores_muestra.shape[1]

fin_primer_tercio = max(
    1,
    numero_dias // 3
)

inicio_ultimo_tercio = max(
    2,
    2 * numero_dias // 3
)

primer_total_directo = (
    valores_muestra[
        :,
        :fin_primer_tercio
    ].sum(axis=1)
)

ultimo_total_directo = (
    valores_muestra[
        :,
        inicio_ultimo_tercio:
    ].sum(axis=1)
)

ratio_log_directo = (
    np.log1p(ultimo_total_directo)
    - np.log1p(primer_total_directo)
)

indice_ratio = NOMBRES_RESUMEN.index(
    "log_ratio_last_first"
)

ratio_log_funcion = resumen_muestra[
    :,
    indice_ratio
]

print("Archivo utilizado:")
print(RUTA_EJEMPLO.name)

print("\nDimensión de la matriz diaria:")
print(valores_muestra.shape)

print("\nDimensión del resumen:")
print(resumen_muestra.shape)

print(
    "\n¿El ratio logarítmico coincide?",
    np.allclose(
        ratio_log_directo,
        ratio_log_funcion
    )
)

print(
    "¿Todos los valores son finitos?",
    np.isfinite(
        resumen_muestra
    ).all()
)

display(resumen_muestra_df)

## Construcción de la matriz de 176 variables

Se define una función que procesa cada archivo de actividad por bloques y transforma los registros diarios en variables conductuales agregadas a nivel de cliente.

Este procesamiento evita cargar simultáneamente en memoria las 65.005 filas y las 90 columnas diarias de cada archivo, lo que reduce el consumo de recursos y permite aplicar el mismo procedimiento de forma consistente a los doce perfiles de actividad.

In [ ]:
def procesar_archivo_actividad(
    ruta_archivo,
    columnas_temporales,
    tamaño_bloque=10_000
):
    """
    Procesa un archivo de actividad por bloques.

    Devuelve una tabla con una fila por cliente y las variables
    conductuales calculadas para el perfil correspondiente.
    """
    nombre_perfil = ruta_archivo.stem
    resultados_bloques = []

    columnas_lectura = [
        "customer_id",
        *columnas_temporales
    ]

    for numero_bloque, bloque in enumerate(
        pd.read_csv(
            ruta_archivo,
            usecols=columnas_lectura,
            chunksize=tamaño_bloque
        ),
        start=1
    ):
        valores = (
            bloque[columnas_temporales]
            .to_numpy(dtype=np.float32)
        )

        resumen = resumir_actividad(valores)

        columnas_resultado = [
            f"{nombre_perfil}__{nombre}"
            for nombre in NOMBRES_RESUMEN
        ]

        resumen_bloque = pd.DataFrame(
            resumen,
            columns=columnas_resultado
        )

        resumen_bloque.insert(
            0,
            "customer_id",
            bloque["customer_id"].to_numpy()
        )

        resultados_bloques.append(resumen_bloque)

        print(
            f"Bloque {numero_bloque:02d} procesado | "
            f"filas: {len(bloque):,}"
        )

    resultado_archivo = pd.concat(
        resultados_bloques,
        ignore_index=True
    )

    return resultado_archivo

### Prueba con un perfil completo

La función se aplica al primer archivo de actividad completo para comprobar que genera una fila por cliente y once variables conductuales. También se verifica que los identificadores de cliente no contienen valores nulos ni duplicados.

In [ ]:
perfil_prueba = archivos_actividad[0]

variables_perfil_prueba = procesar_archivo_actividad(
    ruta_archivo=perfil_prueba,
    columnas_temporales=columnas_ventana,
    tamaño_bloque=10_000
)

print("\nPerfil procesado:")
print(perfil_prueba.stem)

print(
    "Dimensión del resultado:",
    variables_perfil_prueba.shape
)

print(
    "Clientes únicos:",
    variables_perfil_prueba["customer_id"].nunique()
)

print(
    "Duplicados de customer_id:",
    variables_perfil_prueba["customer_id"]
    .duplicated()
    .sum()
)

print(
    "Valores nulos totales:",
    variables_perfil_prueba.isna().sum().sum()
)

display(variables_perfil_prueba.head())

## Construcción de las variables por perfil de actividad

Se procesan de forma secuencial los doce archivos de actividad y se calculan once variables conductuales para cada perfil.

Los resultados se integran con la tabla maestra mediante `customer_id`. Esta correspondencia garantiza que todas las variables queden asociadas al cliente correcto, independientemente del orden original de las filas en cada archivo.

Como resultado, se obtienen 132 variables predictoras: once para cada uno de los doce perfiles de actividad.

In [ ]:
orden_clientes = pd.Index(
    clientes_base["customer_id"],
    name="customer_id"
)

matrices_por_perfil = []

for numero_archivo, ruta_archivo in enumerate(
    archivos_actividad,
    start=1
):
    print("\n" + "=" * 70)
    print(
        f"ARCHIVO {numero_archivo:02d} DE "
        f"{len(archivos_actividad):02d}"
    )
    print(ruta_archivo.name)
    print("=" * 70)

    variables_archivo = procesar_archivo_actividad(
        ruta_archivo=ruta_archivo,
        columnas_temporales=columnas_ventana,
        tamaño_bloque=10_000
    )

    if len(variables_archivo) != len(clientes_base):
        raise ValueError(
            f"{ruta_archivo.name}: número de filas incorrecto. "
            f"Se esperaban {len(clientes_base):,} y se obtuvieron "
            f"{len(variables_archivo):,}."
        )

    if variables_archivo["customer_id"].duplicated().any():
        raise ValueError(
            f"{ruta_archivo.name}: existen customer_id duplicados."
        )

    variables_archivo = (
        variables_archivo
        .set_index("customer_id")
        .reindex(orden_clientes)
    )

    if variables_archivo.isna().any().any():
        raise ValueError(
            f"{ruta_archivo.name}: aparecen valores nulos "
            "después de alinear los clientes."
        )

    variables_archivo = variables_archivo.astype(
        np.float32
    )

    matrices_por_perfil.append(
        variables_archivo
    )

    print(
        f"Perfil completado: {ruta_archivo.stem}"
    )
    print(
        f"Dimensión: {variables_archivo.shape}"
    )

### Integración de las variables de los doce perfiles

Las matrices obtenidas para los doce perfiles de actividad se integran horizontalmente, conservando una única fila por cliente.

El resultado contiene `customer_id` y 132 variables predictoras. En esta etapa no se incorpora la variable objetivo `churn`, lo que evita que intervenga en la construcción de dichas variables y previene posibles fugas de información.

In [ ]:
matriz_variables_perfiles = pd.concat(
    matrices_por_perfil,
    axis=1
)

matriz_variables_perfiles = (
    matriz_variables_perfiles
    .reset_index()
)

numero_variables_perfiles = (
    matriz_variables_perfiles.shape[1] - 1
)

print(
    "Dimensión de la matriz:",
    matriz_variables_perfiles.shape
)

print(
    "Número de clientes:",
    f"{len(matriz_variables_perfiles):,}"
)

print(
    "Número de variables predictoras:",
    numero_variables_perfiles
)

print(
    "Clientes únicos:",
    f"{matriz_variables_perfiles['customer_id'].nunique():,}"
)

print(
    "Identificadores duplicados:",
    matriz_variables_perfiles[
        "customer_id"
    ].duplicated().sum()
)

print(
    "Nombres de columnas duplicados:",
    matriz_variables_perfiles.columns
    .duplicated()
    .sum()
)

print(
    "Valores nulos:",
    matriz_variables_perfiles
    .isna()
    .sum()
    .sum()
)

memoria_mb = (
    matriz_variables_perfiles
    .memory_usage(deep=True)
    .sum()
    / (1024 ** 2)
)

print(
    "Memoria ocupada por la matriz:",
    f"{memoria_mb:,.2f} MB"
)

### Validación de la matriz de variables por perfil

Se comprueba que la matriz contiene los 65.005 clientes esperados y las 132 variables predictoras generadas. Además, se verifica que los identificadores de cliente son únicos y que no existen valores nulos ni columnas duplicadas.

In [ ]:
validacion_matriz_perfiles = {
    "filas_correctas": (
        len(matriz_variables_perfiles) == 65_005
    ),
    "variables_correctas": (
        numero_variables_perfiles == 132
    ),
    "clientes_unicos_correctos": (
        matriz_variables_perfiles[
            "customer_id"
        ].nunique() == 65_005
    ),
    "sin_identificadores_duplicados": (
        matriz_variables_perfiles[
            "customer_id"
        ].duplicated().sum() == 0
    ),
    "sin_columnas_duplicadas": (
        matriz_variables_perfiles.columns
        .duplicated()
        .sum() == 0
    ),
    "sin_valores_nulos": (
        matriz_variables_perfiles
        .isna()
        .sum()
        .sum() == 0
    )
}

validacion_matriz_perfiles_df = pd.DataFrame({
    "comprobacion": validacion_matriz_perfiles.keys(),
    "resultado": validacion_matriz_perfiles.values()
})

display(validacion_matriz_perfiles_df)

print(
    "¿La matriz de variables por perfil es válida?",
    all(validacion_matriz_perfiles.values())
)

### Almacenamiento de la matriz de variables por perfil

La matriz validada se guarda en formato `pickle` como punto de control intermedio.

Este formato conserva los tipos de datos y permite recuperar la matriz con rapidez, evitando repetir el procesamiento de los doce archivos de actividad si se reinicia la sesión de Jupyter.

In [ ]:
RUTA_MATRIZ_PERFILES = (
    RUTA_RESULTADOS
    / "variables_perfiles_90d.pkl"
)

matriz_variables_perfiles.to_pickle(
    RUTA_MATRIZ_PERFILES
)

RUTA_COLUMNAS_PERFILES = (
    RUTA_RESULTADOS
    / "columnas_variables_perfiles_90d.csv"
)

pd.DataFrame({
    "variable": matriz_variables_perfiles
    .columns
    .drop("customer_id")
}).to_csv(
    RUTA_COLUMNAS_PERFILES,
    index=False
)

print("Matriz guardada en:")
print(RUTA_MATRIZ_PERFILES)

print("\nInventario de variables guardado en:")
print(RUTA_COLUMNAS_PERFILES)

print(
    "\n¿Existe la matriz?",
    RUTA_MATRIZ_PERFILES.exists()
)

print(
    "Tamaño de la matriz:",
    f"{RUTA_MATRIZ_PERFILES.stat().st_size / (1024 ** 2):,.2f} MB"
)

print(
    "Variables registradas:",
    pd.read_csv(RUTA_COLUMNAS_PERFILES).shape[0]
)

## Construcción de variables agregadas por categoría

Los doce perfiles de actividad se agrupan en tres categorías funcionales: aplicaciones, voz y paquetes. Además, se construye un agregado global que resume la actividad de todos los perfiles.

Para cada categoría se suman las secuencias diarias de los perfiles que la componen y, sobre la serie resultante, se calculan las mismas once variables conductuales definidas anteriormente. El agregado global se utiliza como una señal compuesta de actividad y no debe interpretarse como una magnitud física homogénea.

In [ ]:
categorias_actividad = {
    "apps": [
        ruta
        for ruta in archivos_actividad
        if ruta.stem.startswith("usage_app_")
    ],
    "voice": [
        ruta
        for ruta in archivos_actividad
        if ruta.stem.startswith("usage_voice_")
    ],
    "packages": [
        ruta
        for ruta in archivos_actividad
        if ruta.stem.startswith("usage_pack_")
    ]
}

resumen_categorias = pd.DataFrame({
    "categoria": categorias_actividad.keys(),
    "numero_perfiles": [
        len(rutas)
        for rutas in categorias_actividad.values()
    ],
    "perfiles": [
        [ruta.stem for ruta in rutas]
        for rutas in categorias_actividad.values()
    ]
})

display(resumen_categorias)

rutas_asignadas = [
    ruta
    for rutas in categorias_actividad.values()
    for ruta in rutas
]

print("Perfiles totales asignados:", len(rutas_asignadas))
print("Perfiles únicos asignados:", len(set(rutas_asignadas)))

print(
    "¿Todos los archivos están asignados una sola vez?",
    (
        len(rutas_asignadas) == 12
        and len(set(rutas_asignadas)) == 12
        and set(rutas_asignadas) == set(archivos_actividad)
    )
)

### Procesamiento eficiente de las categorías

Los archivos de actividad se procesan por bloques, de modo que los datos de cada perfil contribuyen tanto a su categoría funcional como al agregado global.

Este procedimiento evita realizar lecturas adicionales y reduce el consumo de memoria, ya que durante el procesamiento solo se mantienen los datos necesarios para construir el agregado global y el correspondiente a la categoría analizada.

In [ ]:
import gc


def convertir_resumen_en_dataframe(
    nombre_grupo,
    valores_diarios,
    indice_clientes
):
    """
    Aplica las once funciones conductuales y devuelve
    una tabla indexada por customer_id.
    """
    resumen = resumir_actividad(valores_diarios)

    columnas = [
        f"agg_{nombre_grupo}__{nombre}"
        for nombre in NOMBRES_RESUMEN
    ]

    resultado = pd.DataFrame(
        resumen,
        index=indice_clientes,
        columns=columnas
    )

    return resultado.astype(np.float32)


numero_clientes = len(orden_clientes)
numero_dias = len(columnas_ventana)

posicion_por_cliente = pd.Series(
    np.arange(numero_clientes, dtype=np.int64),
    index=orden_clientes
)

acumulado_global = np.zeros(
    (numero_clientes, numero_dias),
    dtype=np.float32
)

matrices_agregadas = {}

for nombre_categoria, rutas_categoria in categorias_actividad.items():

    print("\n" + "=" * 70)
    print(f"CATEGORÍA: {nombre_categoria.upper()}")
    print("=" * 70)

    acumulado_categoria = np.zeros(
        (numero_clientes, numero_dias),
        dtype=np.float32
    )

    for numero_archivo, ruta_archivo in enumerate(
        rutas_categoria,
        start=1
    ):
        print(
            f"\nArchivo {numero_archivo} de "
            f"{len(rutas_categoria)}: {ruta_archivo.name}"
        )

        clientes_vistos_archivo = np.zeros(
            numero_clientes,
            dtype=bool
        )

        for numero_bloque, bloque in enumerate(
            pd.read_csv(
                ruta_archivo,
                usecols=[
                    "customer_id",
                    *columnas_ventana
                ],
                chunksize=10_000
            ),
            start=1
        ):
            posiciones = posicion_por_cliente.reindex(
                bloque["customer_id"]
            )

            if posiciones.isna().any():
                raise ValueError(
                    f"{ruta_archivo.name}: existen clientes "
                    "que no aparecen en la tabla maestra."
                )

            posiciones = posiciones.to_numpy(
                dtype=np.int64
            )

            if len(np.unique(posiciones)) != len(posiciones):
                raise ValueError(
                    f"{ruta_archivo.name}: existen clientes "
                    "duplicados dentro de un bloque."
                )

            if clientes_vistos_archivo[posiciones].any():
                raise ValueError(
                    f"{ruta_archivo.name}: existen clientes "
                    "repetidos entre bloques."
                )

            valores = (
                bloque[columnas_ventana]
                .to_numpy(dtype=np.float32)
            )

            if not np.isfinite(valores).all():
                raise ValueError(
                    f"{ruta_archivo.name}: existen valores "
                    "no finitos."
                )

            acumulado_categoria[posiciones] += valores
            acumulado_global[posiciones] += valores

            clientes_vistos_archivo[posiciones] = True

            print(
                f"  Bloque {numero_bloque:02d} | "
                f"{len(bloque):,} filas"
            )

        if not clientes_vistos_archivo.all():
            clientes_ausentes = (
                ~clientes_vistos_archivo
            ).sum()

            raise ValueError(
                f"{ruta_archivo.name}: faltan "
                f"{clientes_ausentes:,} clientes."
            )

    matrices_agregadas[nombre_categoria] = (
        convertir_resumen_en_dataframe(
            nombre_grupo=nombre_categoria,
            valores_diarios=acumulado_categoria,
            indice_clientes=orden_clientes
        )
    )

    print(
        f"\nCategoría {nombre_categoria} completada:",
        matrices_agregadas[nombre_categoria].shape
    )

    del acumulado_categoria
    gc.collect()


matrices_agregadas["global"] = (
    convertir_resumen_en_dataframe(
        nombre_grupo="global",
        valores_diarios=acumulado_global,
        indice_clientes=orden_clientes
    )
)

print(
    "\nAgregado global completado:",
    matrices_agregadas["global"].shape
)

del acumulado_global
gc.collect()

### Integración y validación de las variables agregadas

Los cuatro bloques de variables se integran horizontalmente mediante `customer_id`, conservando una única fila por cliente.

Se comprueba que el resultado contiene los 65.005 clientes esperados y 44 variables predictoras: once correspondientes a aplicaciones, once a voz, once a paquetes y once al agregado global.

In [ ]:
orden_grupos_agregados = [
    "global",
    "apps",
    "voice",
    "packages"
]

matriz_variables_agregadas = pd.concat(
    [
        matrices_agregadas[grupo]
        for grupo in orden_grupos_agregados
    ],
    axis=1
).reset_index()

numero_variables_agregadas = (
    matriz_variables_agregadas.shape[1] - 1
)

valores_agregados = (
    matriz_variables_agregadas
    .drop(columns="customer_id")
    .to_numpy()
)

print(
    "Dimensión:",
    matriz_variables_agregadas.shape
)

print(
    "Variables agregadas:",
    numero_variables_agregadas
)

print(
    "Clientes únicos:",
    matriz_variables_agregadas[
        "customer_id"
    ].nunique()
)

print(
    "Identificadores duplicados:",
    matriz_variables_agregadas[
        "customer_id"
    ].duplicated().sum()
)

print(
    "Columnas duplicadas:",
    matriz_variables_agregadas.columns
    .duplicated()
    .sum()
)

print(
    "Valores nulos:",
    matriz_variables_agregadas
    .isna()
    .sum()
    .sum()
)

print(
    "¿Todos los valores numéricos son finitos?",
    np.isfinite(valores_agregados).all()
)

print(
    "¿La matriz agregada es válida?",
    (
        matriz_variables_agregadas.shape == (65_005, 45)
        and numero_variables_agregadas == 44
        and matriz_variables_agregadas[
            "customer_id"
        ].nunique() == 65_005
        and matriz_variables_agregadas[
            "customer_id"
        ].duplicated().sum() == 0
        and matriz_variables_agregadas.columns
        .duplicated().sum() == 0
        and matriz_variables_agregadas
        .isna().sum().sum() == 0
        and np.isfinite(valores_agregados).all()
    )
)

## Construcción de la matriz final de 176 variables

Las 132 variables específicas de los perfiles y las 44 variables agregadas se integran mediante una unión uno a uno por `customer_id`.

La matriz resultante contiene exclusivamente las 176 variables predictoras. La variable objetivo `churn` y la partición original se incorporarán en una etapa posterior, evitando así que intervengan accidentalmente en el proceso de ingeniería de variables.

In [ ]:
matriz_variables_176 = (
    matriz_variables_perfiles
    .merge(
        matriz_variables_agregadas,
        on="customer_id",
        how="inner",
        validate="one_to_one"
    )
)

columnas_predictoras_176 = [
    columna
    for columna in matriz_variables_176.columns
    if columna != "customer_id"
]

valores_176 = matriz_variables_176[
    columnas_predictoras_176
].to_numpy()

print(
    "Dimensión de la matriz:",
    matriz_variables_176.shape
)

print(
    "Número de variables predictoras:",
    len(columnas_predictoras_176)
)

print(
    "Clientes únicos:",
    matriz_variables_176[
        "customer_id"
    ].nunique()
)

print(
    "Identificadores duplicados:",
    matriz_variables_176[
        "customer_id"
    ].duplicated().sum()
)

print(
    "Columnas duplicadas:",
    matriz_variables_176.columns
    .duplicated().sum()
)

print(
    "Valores nulos:",
    matriz_variables_176
    .isna()
    .sum()
    .sum()
)

print(
    "¿Todos los valores son finitos?",
    np.isfinite(valores_176).all()
)

print(
    "¿La matriz de 176 variables es válida?",
    (
        matriz_variables_176.shape == (65_005, 177)
        and len(columnas_predictoras_176) == 176
        and matriz_variables_176[
            "customer_id"
        ].nunique() == 65_005
        and matriz_variables_176[
            "customer_id"
        ].duplicated().sum() == 0
        and matriz_variables_176.columns
        .duplicated().sum() == 0
        and matriz_variables_176
        .isna().sum().sum() == 0
        and np.isfinite(valores_176).all()
    )
)

## Incorporación de la variable objetivo y la partición

Una vez construida y validada la matriz de variables predictoras, se incorporan la variable objetivo `churn` y la partición original de cada cliente.

La tabla resultante contiene una fila por cliente, 176 variables predictoras, la variable objetivo y un indicador que identifica su pertenencia al conjunto de entrenamiento o de prueba.

In [ ]:
dataset_modelado_90d = (
    clientes_base
    .merge(
        matriz_variables_176,
        on="customer_id",
        how="inner",
        validate="one_to_one"
    )
)

print(
    "Dimensión final:",
    dataset_modelado_90d.shape
)

print(
    "Variables predictoras:",
    len(columnas_predictoras_176)
)

print("\nDistribución por partición:")
print(
    dataset_modelado_90d[
        "particion"
    ].value_counts()
)

print(
    "\nValores nulos totales:",
    dataset_modelado_90d
    .isna()
    .sum()
    .sum()
)

print(
    "¿churn está excluida de las variables predictoras?",
    "churn" not in columnas_predictoras_176
)

print(
    "¿particion está excluida de las variables predictoras?",
    "particion" not in columnas_predictoras_176
)

print(
    "¿El dataset de modelado es válido?",
    (
        dataset_modelado_90d.shape == (65_005, 179)
        and len(columnas_predictoras_176) == 176
        and dataset_modelado_90d[
            "customer_id"
        ].nunique() == 65_005
        and dataset_modelado_90d
        .isna().sum().sum() == 0
        and "churn" not in columnas_predictoras_176
        and "particion" not in columnas_predictoras_176
    )
)

### Almacenamiento del dataset preparado

Las matrices validadas se guardan en formato `pickle` para conservar los tipos de datos y evitar repetir el proceso de ingeniería de variables.

Además, se almacena en formato CSV un inventario con los nombres de las 176 variables predictoras.

In [ ]:
RUTA_AGREGADAS_90D = (
    RUTA_RESULTADOS
    / "variables_agregadas_90d.pkl"
)

RUTA_VARIABLES_176_90D = (
    RUTA_RESULTADOS
    / "matriz_variables_176_90d.pkl"
)

RUTA_DATASET_MODELADO_90D = (
    RUTA_RESULTADOS
    / "dataset_modelado_90d.pkl"
)

RUTA_COLUMNAS_176 = (
    RUTA_RESULTADOS
    / "columnas_predictoras_176_90d.csv"
)

matriz_variables_agregadas.to_pickle(
    RUTA_AGREGADAS_90D
)

matriz_variables_176.to_pickle(
    RUTA_VARIABLES_176_90D
)

dataset_modelado_90d.to_pickle(
    RUTA_DATASET_MODELADO_90D
)

pd.DataFrame({
    "orden": range(1, 177),
    "variable": columnas_predictoras_176
}).to_csv(
    RUTA_COLUMNAS_176,
    index=False
)

archivos_guardados = [
    RUTA_AGREGADAS_90D,
    RUTA_VARIABLES_176_90D,
    RUTA_DATASET_MODELADO_90D,
    RUTA_COLUMNAS_176
]

for ruta in archivos_guardados:
    print(
        f"{ruta.name:<40} | "
        f"existe: {ruta.exists()} | "
        f"{ruta.stat().st_size / (1024 ** 2):,.2f} MB"
    )

print(
    "\nVariables registradas:",
    pd.read_csv(RUTA_COLUMNAS_176).shape[0]
)

print(
    "¿Todos los archivos se guardaron?",
    all(ruta.exists() for ruta in archivos_guardados)
)

### Verificación final del esquema de variables regenerado

Una vez repetido el proceso de ingeniería de variables con la definición corregida, se comprueba que los archivos almacenados contienen el esquema previsto.

La matriz combina doce perfiles individuales, tres categorías funcionales y un agregado global. En total, se consideran dieciséis ámbitos de actividad y se calculan once variables conductuales para cada uno, lo que genera 176 variables predictoras.

La validación comprueba que `log_ratio_last_first` esté presente una vez por cada ámbito, que la variable sustituida `last_third_total` no figure en el inventario y que cada una de las once variables conductuales aparezca exactamente dieciséis veces.

Esta comprobación permite detectar esquemas desactualizados que podrían conservar el número correcto de columnas, pero incluir definiciones incorrectas. El resultado esperado es un inventario de 176 variables, con dieciséis variantes de `log_ratio_last_first`, ninguna de `last_third_total` y una validación global positiva.

In [ ]:
inventario_176 = pd.read_csv(
    RUTA_COLUMNAS_176
)

conteo_por_indicador = (
    inventario_176["variable"]
    .str.rsplit("__", n=1)
    .str[-1]
    .value_counts()
    .reindex(NOMBRES_RESUMEN)
)

numero_ratio_logaritmico = (
    inventario_176["variable"]
    .str.endswith("__log_ratio_last_first")
    .sum()
)

numero_variables_antiguas = (
    inventario_176["variable"]
    .str.contains(
        "last_third_total",
        regex=False
    )
    .sum()
)

display(
    conteo_por_indicador
    .rename_axis("indicador")
    .reset_index(name="numero_ambitos")
)

print(
    "Variables totales:",
    len(inventario_176)
)

print(
    "Variables log_ratio_last_first:",
    numero_ratio_logaritmico
)

print(
    "Variables last_third_total:",
    numero_variables_antiguas
)

esquema_regenerado_valido = (
    len(inventario_176) == 176
    and numero_ratio_logaritmico == 16
    and numero_variables_antiguas == 0
    and conteo_por_indicador.notna().all()
    and conteo_por_indicador.eq(16).all()
)

print(
    "¿El esquema regenerado es válido?",
    esquema_regenerado_valido
)

## Construcción y depuración de las variables candidatas

La matriz validada contiene 176 variables conductuales construidas a partir de doce perfiles individuales, tres categorías funcionales y un agregado global. Para documentar el proceso completo de selección, se incorporan inicialmente dos variables adicionales relacionadas con la diversidad de uso.

La variable `diversity__active_usage_types` indica el número de perfiles individuales que presentan actividad positiva durante la ventana de observación, mientras que `diversity__active_apps` contabiliza cuántas de las seis aplicaciones disponibles registran actividad.

Tras incorporar estas dos variables, se obtiene un conjunto inicial de 178 variables candidatas. A continuación, se analiza el número de valores únicos de cada columna para identificar posibles variables constantes.

Una variable constante presenta el mismo valor para todos los clientes, por lo que no permite diferenciarlos ni aporta información útil para el modelado. Por este motivo, estas variables se excluyen del conjunto definitivo.

Se espera identificar dos variables constantes y conservar 176 variables seleccionadas. Finalmente, se comprueba que el esquema resultante coincide exactamente con el de la matriz de 176 variables validada previamente.

In [ ]:
# Identificación de las columnas de actividad total
# correspondientes a los doce perfiles individuales.

columnas_total_perfiles = [
    columna
    for columna in columnas_predictoras_176
    if (
        columna.startswith("usage_")
        and columna.endswith("__total")
    )
]

columnas_total_apps = [
    columna
    for columna in columnas_total_perfiles
    if columna.startswith("usage_app_")
]

print(
    "Columnas totales de perfiles encontradas:",
    len(columnas_total_perfiles)
)

print(
    "Columnas totales de aplicaciones encontradas:",
    len(columnas_total_apps)
)

if len(columnas_total_perfiles) != 12:
    raise ValueError(
        "Se esperaban 12 columnas de actividad total "
        "correspondientes a los perfiles individuales."
    )

if len(columnas_total_apps) != 6:
    raise ValueError(
        "Se esperaban 6 columnas de actividad total "
        "correspondientes a aplicaciones."
    )


# Creación de la matriz candidata.

matriz_variables_178 = matriz_variables_176.copy()

matriz_variables_178[
    "diversity__active_usage_types"
] = (
    matriz_variables_178[
        columnas_total_perfiles
    ]
    .gt(0)
    .sum(axis=1)
    .astype(np.int16)
)

matriz_variables_178[
    "diversity__active_apps"
] = (
    matriz_variables_178[
        columnas_total_apps
    ]
    .gt(0)
    .sum(axis=1)
    .astype(np.int16)
)

columnas_candidatas_178 = [
    columna
    for columna in matriz_variables_178.columns
    if columna != "customer_id"
]


# Diagnóstico de variabilidad de las columnas candidatas.

diagnostico_variables_178 = pd.DataFrame({
    "variable": columnas_candidatas_178,
    "valores_unicos": [
        matriz_variables_178[columna].nunique(
            dropna=False
        )
        for columna in columnas_candidatas_178
    ]
})

diagnostico_variables_178["constante"] = (
    diagnostico_variables_178[
        "valores_unicos"
    ] == 1
)

variables_constantes = (
    diagnostico_variables_178.loc[
        diagnostico_variables_178["constante"],
        "variable"
    ]
    .tolist()
)

variables_seleccionadas = (
    diagnostico_variables_178.loc[
        ~diagnostico_variables_178["constante"],
        "variable"
    ]
    .tolist()
)


# Reconstrucción de la matriz después de eliminar constantes.

matriz_variables_seleccionadas = (
    matriz_variables_178[
        [
            "customer_id",
            *variables_seleccionadas
        ]
    ]
    .copy()
)

coincidencia_esquema = (
    variables_seleccionadas
    == columnas_predictoras_176
)


# Presentación de resultados.

print(
    "\nVariables candidatas:",
    len(columnas_candidatas_178)
)

print(
    "Variables constantes:",
    len(variables_constantes)
)

print(
    "Variables seleccionadas:",
    len(variables_seleccionadas)
)

print("\nVariables constantes detectadas:")

display(
    diagnostico_variables_178[
        diagnostico_variables_178["constante"]
    ]
)

print(
    "Valores únicos de active_usage_types:",
    sorted(
        matriz_variables_178[
            "diversity__active_usage_types"
        ].unique()
    )
)

print(
    "Valores únicos de active_apps:",
    sorted(
        matriz_variables_178[
            "diversity__active_apps"
        ].unique()
    )
)

print(
    "\n¿El esquema seleccionado coincide con "
    "las 176 variables previamente validadas?",
    coincidencia_esquema
)

seleccion_variables_valida = (
    len(columnas_candidatas_178) == 178
    and len(variables_constantes) == 2
    and set(variables_constantes) == {
        "diversity__active_usage_types",
        "diversity__active_apps"
    }
    and len(variables_seleccionadas) == 176
    and matriz_variables_seleccionadas.shape
        == (65_005, 177)
    and coincidencia_esquema
)

print(
    "¿La selección de variables es válida?",
    seleccion_variables_valida
)

### Almacenamiento de la matriz candidata y del diagnóstico de selección

Una vez identificadas las 178 variables candidatas y confirmado que dos de ellas son constantes, se almacenan los resultados de esta fase para garantizar la trazabilidad del proceso de selección.

Se guardan los siguientes elementos:

1. La matriz completa con las 178 variables candidatas, antes de excluir las variables constantes.
2. El diagnóstico de variabilidad de cada columna, que incluye el número de valores únicos y su clasificación como variable constante o no constante.
3. Un resumen estructurado con el número inicial de variables candidatas, las variables excluidas y el número final de variables seleccionadas.

Estos archivos permiten verificar que la reducción de 178 a 176 variables responde a un criterio reproducible: la exclusión de columnas sin variación entre clientes.

Para el modelado se mantiene la matriz de 176 variables validada previamente. La matriz de 178 variables se conserva únicamente como evidencia técnica del proceso de selección.

In [ ]:
import json


# ---------------------------------------------------------
# Preparación del diagnóstico completo
# ---------------------------------------------------------

diagnostico_variables_178["seleccionada"] = (
    ~diagnostico_variables_178["constante"]
)

diagnostico_variables_178["motivo"] = np.where(
    diagnostico_variables_178["constante"],
    "excluded_constant",
    "retained"
)


# ---------------------------------------------------------
# Definición de las rutas de salida
# ---------------------------------------------------------

RUTA_MATRIZ_CANDIDATA_178 = (
    RUTA_RESULTADOS
    / "matriz_variables_candidatas_178_90d.pkl"
)

RUTA_DIAGNOSTICO_178 = (
    RUTA_RESULTADOS
    / "diagnostico_variables_178_90d.csv"
)

RUTA_RESUMEN_SELECCION_178 = (
    RUTA_RESULTADOS
    / "resumen_seleccion_variables_178_90d.json"
)


# ---------------------------------------------------------
# Almacenamiento de los archivos
# ---------------------------------------------------------

matriz_variables_178.to_pickle(
    RUTA_MATRIZ_CANDIDATA_178
)

diagnostico_variables_178.to_csv(
    RUTA_DIAGNOSTICO_178,
    index=False
)

resumen_seleccion_178 = {
    "window_days": 90,
    "candidate_features": len(
        columnas_candidatas_178
    ),
    "constant_features": len(
        variables_constantes
    ),
    "selected_features": len(
        variables_seleccionadas
    ),
    "excluded_features": variables_constantes,
    "selection_criterion": "excluded_constant",
    "matches_validated_176_schema": bool(
        coincidencia_esquema
    )
}

with open(
    RUTA_RESUMEN_SELECCION_178,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        resumen_seleccion_178,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Comprobación de los archivos generados
# ---------------------------------------------------------

archivos_seleccion = [
    RUTA_MATRIZ_CANDIDATA_178,
    RUTA_DIAGNOSTICO_178,
    RUTA_RESUMEN_SELECCION_178
]

for ruta in archivos_seleccion:
    print(
        f"{ruta.name:<50} | "
        f"existe: {ruta.exists()} | "
        f"tamaño: "
        f"{ruta.stat().st_size / (1024 ** 2):,.2f} MB"
    )

print(
    "\nVariables candidatas guardadas:",
    matriz_variables_178.shape[1] - 1
)

print(
    "Filas del diagnóstico:",
    len(
        pd.read_csv(
            RUTA_DIAGNOSTICO_178
        )
    )
)

with open(
    RUTA_RESUMEN_SELECCION_178,
    "r",
    encoding="utf-8"
) as archivo:
    resumen_guardado = json.load(archivo)

print(
    "Variables excluidas registradas:",
    resumen_guardado["excluded_features"]
)

print(
    "¿Todos los archivos se guardaron?",
    all(
        ruta.exists()
        for ruta in archivos_seleccion
    )
)

print(
    "¿El resumen guardado es coherente?",
    (
        resumen_guardado[
            "candidate_features"
        ] == 178
        and resumen_guardado[
            "constant_features"
        ] == 2
        and resumen_guardado[
            "selected_features"
        ] == 176
        and set(
            resumen_guardado[
                "excluded_features"
            ]
        ) == {
            "diversity__active_usage_types",
            "diversity__active_apps"
        }
        and resumen_guardado[
            "matches_validated_176_schema"
        ] is True
    )
)

## Cierre de la preparación del dataset

En este notebook se ha transformado la información diaria original en un dataset analítico con una única fila por cliente.

Durante el proceso se han conservado las particiones originales de entrenamiento y prueba, se han alineado los registros mediante `customer_id` y se han construido las variables predictoras exclusivamente a partir de la actividad observada, sin utilizar la variable objetivo `churn` durante la ingeniería de variables.

Para cada uno de los doce perfiles individuales, las tres categorías funcionales y el agregado global se han calculado once variables conductuales. Como resultado, se ha obtenido un esquema con 176 variables predictoras válidas.

Además, se ha reconstruido el conjunto inicial de 178 variables candidatas mediante la incorporación de dos variables de diversidad. Ambas han sido identificadas como constantes y excluidas mediante un criterio automático y reproducible, por lo que el conjunto definitivo conserva las 176 variables que presentan variación entre clientes.

Como cierre de esta fase, se realiza una validación conjunta de los resultados almacenados. Esta comprobación verifica la existencia de los archivos, las dimensiones de las matrices, el número de clientes, la ausencia de valores nulos, la separación entre las variables predictoras y la variable objetivo, y la coherencia del proceso de selección.

Una vez superadas estas comprobaciones, el dataset preparado puede utilizarse como una entrada estable para las siguientes fases de análisis, comparación temporal y modelado.

In [ ]:
# Archivos principales generados durante la preparación

archivos_finales_preparacion = {
    "clientes_base": RUTA_CLIENTES_BASE,
    "variables_perfiles": RUTA_MATRIZ_PERFILES,
    "variables_agregadas": RUTA_AGREGADAS_90D,
    "matriz_176": RUTA_VARIABLES_176_90D,
    "dataset_modelado": RUTA_DATASET_MODELADO_90D,
    "inventario_176": RUTA_COLUMNAS_176,
    "matriz_candidata_178": RUTA_MATRIZ_CANDIDATA_178,
    "diagnostico_178": RUTA_DIAGNOSTICO_178,
    "resumen_seleccion_178": RUTA_RESUMEN_SELECCION_178
}

resumen_archivos_finales = pd.DataFrame({
    "archivo": archivos_finales_preparacion.keys(),
    "ruta": [
        str(ruta)
        for ruta in archivos_finales_preparacion.values()
    ],
    "existe": [
        ruta.exists()
        for ruta in archivos_finales_preparacion.values()
    ],
    "tamaño_mb": [
        round(
            ruta.stat().st_size / (1024 ** 2),
            2
        )
        if ruta.exists()
        else np.nan
        for ruta in archivos_finales_preparacion.values()
    ]
})

display(resumen_archivos_finales)


# Validaciones finales del contenido en memoria

comprobaciones_finales = {
    "clientes_totales_correctos": (
        len(clientes_base) == 65_005
    ),
    "clientes_unicos_correctos": (
        clientes_base["customer_id"].nunique()
        == 65_005
    ),
    "matriz_176_dimensiones_correctas": (
        matriz_variables_176.shape
        == (65_005, 177)
    ),
    "dataset_modelado_dimensiones_correctas": (
        dataset_modelado_90d.shape
        == (65_005, 179)
    ),
    "variables_predictoras_correctas": (
        len(columnas_predictoras_176) == 176
    ),
    "sin_nulos_dataset_modelado": (
        dataset_modelado_90d
        .isna()
        .sum()
        .sum() == 0
    ),
    "churn_fuera_de_predictores": (
        "churn"
        not in columnas_predictoras_176
    ),
    "particion_fuera_de_predictores": (
        "particion"
        not in columnas_predictoras_176
    ),
    "esquema_regenerado_valido": (
        esquema_regenerado_valido
    ),
    "seleccion_178_a_176_valida": (
        seleccion_variables_valida
    ),
    "archivos_generados_correctamente": (
        resumen_archivos_finales[
            "existe"
        ].all()
    )
}

comprobaciones_finales_df = pd.DataFrame({
    "comprobacion": comprobaciones_finales.keys(),
    "resultado": comprobaciones_finales.values()
})

display(comprobaciones_finales_df)

preparacion_completada = all(
    comprobaciones_finales.values()
)

print(
    "¿La preparación del dataset ha finalizado "
    "correctamente?",
    preparacion_completada
)

print(
    "\nClientes preparados:",
    f"{len(clientes_base):,}"
)

print(
    "Variables candidatas:",
    len(columnas_candidatas_178)
)

print(
    "Variables seleccionadas:",
    len(columnas_predictoras_176)
)

print(
    "Partición de entrenamiento:",
    (
        clientes_base["particion"]
        == "train"
    ).sum()
)

print(
    "Partición de prueba:",
    (
        clientes_base["particion"]
        == "test"
    ).sum()
)